In [1]:
import sys

sys.path.append("../src")

In [2]:
import os
import shutil

import optuna
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from tqdm.notebook import tqdm

from experiments.ensemble_evaluation import EnsembleEvaluation
from experiments.evaluation_run import EvaluationRun
from utils import *

In [3]:
paths = Paths("../configs/paths.json")

In [4]:
def get_best_trial_number(fold_dir):
    path = os.path.join(fold_dir, "optuna.db")
    study = optuna.load_study(study_name=None, storage=f"sqlite:///{path}")
    return study.best_trial.number

In [8]:
import numpy as np
f = np.load("/media/ammar/New Volume/Engagement Data/Features/118/118_450_4_4_1_3.npz")

In [10]:
list(f.keys())

['embedding',
 'expression',
 'valence_arousal',
 'landmarks',
 'pose',
 'OpenFace_gaze',
 'OpenFace_blink',
 'OpenFace_au_r',
 'OpenFace_au_c',
 'OpenFace_pose',
 'EEG_BetaL',
 'EEG_BetaH',
 'EEG_Alpha',
 'EEG_Theta',
 'EEG_Gamma',
 'GP_Fixation',
 'GP_gaze_pog',
 'GP_Pupil_pose',
 'GP_PupilDiameter',
 'GP_Blink',
 'HR_BPM',
 'HR_RR_Intervals',
 'EEG_eng_index',
 'GP_Saccade',
 'EEG_index1',
 'EEG_index2',
 'EEG_index3',
 'EEG_index4',
 'EEG_BetaL_100ms',
 'EEG_BetaH_100ms',
 'EEG_Alpha_100ms',
 'EEG_Theta_100ms',
 'EEG_Gamma_100ms',
 'EEG_index1_100ms',
 'EEG_index2_100ms',
 'EEG_index3_100ms',
 'EEG_index4_100ms',
 'EEG_eng_index_100ms',
 'GP_Fixation_100ms',
 'GP_gaze_pog_100ms',
 'GP_Pupil_pose_100ms',
 'GP_PupilDiameter_100ms',
 'GP_Blink_100ms',
 'GP_Saccade_100ms',
 'HR_BPM_100ms',
 'HR_RR_Intervals_100ms']

In [15]:
f["OpenFace_pose"][::3].shape

(350, 6)

In [12]:
f["GP_Fixation_100ms"].shape

(390, 3)

In [11]:
f["EEG_BetaL"].shape

(386, 14)

In [6]:
# # save best trials
# for f in range(7):
#     fold_root = Path(f"../results/optimization/BCE/fold_{f}")
#     best_trial = get_best_trial_number(fold_root)
#     path = fold_root / f"trial_{best_trial}"
#     shutil.copytree(path, f"../results/best/fold_{f}")

In [7]:
# fold = "fold_1"
# # path = Path(f"../results/optimization/BCE/{fold}/")
# # tn = get_best_trial_number(path)
# # config = read_yaml_config(path / f"trial_{tn}" / "config.yml")
# config = read_yaml_config("../configs/training_configs.yml")
# config["labels_mapper"] = None
# config["splits_path"] = f"../configs/splits/{fold}"
# config["augmentations"] = [["mirror", True], ["shift", True]]
# config["selected_features"][0] = "embedding"
# config["class_weights"] = None
# with open("../results/config.yml", "w") as f:
#     yaml.dump(config, f)
# config

In [8]:
# read_yaml_config("../results/config.yml")

In [8]:
fold = "fold_6#HP"
tn = get_best_trial_number(f"../results/optimization/{fold}/")
show_results(f"../results/optimization/{fold}/trial_{tn}/models", suffix="val")

[I 2025-01-03 08:17:38,602] Study name was omitted but trying to load 'webcam_engagement_classifier' because that was the only study found in the storage.


Accuracy: 0.7008142827637797 ± 0.045202804753364495


,Low,High,Avg.
metric,,,
precision,0.43 ± 0.27,0.8 ± 0.07,0.62 ± 0.12
recall,0.45 ± 0.21,0.79 ± 0.08,0.62 ± 0.11
f1-score,0.42 ± 0.22,0.79 ± 0.04,0.61 ± 0.11
support,28.86 ± 10.61,79.57 ± 19.48,108.43 ± 18.27


In [10]:
def get_fold_results(fold, best_trial=True, root="../results/optimization"):
    if fold == 6:
        fold = f"fold_{fold}#HP"
    else:
        fold = f"fold_{fold}"
    if best_trial:
        tn = get_best_trial_number(f"{root}/{fold}/")
        experiment_root = f"{root}/{fold}/trial_{tn}"
    else:
        experiment_root = f"{root}"

    eval_obj = EnsembleEvaluation(
        data_root=paths["FeaturesRoot"],
        models_root=f"{experiment_root}/models",
        config_path=f"{experiment_root}/config.yml",
        splits_root=f"../configs/splits/{fold}/",
        eval_split="test",
    )
    df = eval_obj.generate_ensemble_predictions()
    y_true = df.y_true_binary.astype(int)
    results = []
    for mode in ["soft", "hard"]:#, "weighted"]:
        y_prob = df[mode]
        y_pred = (y_prob.apply(lambda x: x > 0.5)).astype(int)
        report = pd.DataFrame(
            classification_report(y_true, y_pred, output_dict=True)
        ).assign(roc_auc=roc_auc_score(y_true, y_prob))
        results.append(report.assign(fold=fold))
    return results

In [16]:
results = []
for fold in tqdm(range(7), leave=False):
    results.append(get_fold_results(fold)[0])

In [24]:
all_results = pd.concat(results).reset_index().rename(columns={"index":"Metric"}).drop("fold", axis=1)
all_results

,Metric,0,1,accuracy,macro avg,weighted avg,roc_auc
0,precision,0.250000,0.766667,0.719697,0.508333,0.645328,0.667518
1,recall,0.096774,0.910891,0.719697,0.503833,0.719697,0.667518
2,f1-score,0.139535,0.832579,0.719697,0.486057,0.669819,0.667518
3,support,31.000000,101.000000,0.719697,132.000000,132.000000,0.667518
4,precision,0.326087,0.896552,0.699248,0.611319,0.793611,0.722859
5,recall,0.625000,0.715596,0.699248,0.670298,0.699248,0.722859
6,f1-score,0.428571,0.795918,0.699248,0.612245,0.729630,0.722859
7,support,24.000000,109.000000,0.699248,133.000000,133.000000,0.722859
8,precision,0.383333,0.770270,0.597015,0.576802,0.654767,0.602394
9,recall,0.575000,0.606383,0.597015,0.590691,0.597015,0.602394


In [25]:
avg_df = all_results.groupby("Metric").mean()
std_df = all_results.groupby("Metric").std()
combine_avg_std_tables_latex(avg_df, std_df)

Accuracy: 0.6221986972765746 ± 0.09274854471938608
ROC-AUC: 0.6175306555667026 ± 0.09038268254757874


,Low,High,Avg.
Metric,,,
precision,0.34 ± 0.14,0.74 ± 0.11,0.54 ± 0.08
recall,0.39 ± 0.2,0.71 ± 0.11,0.55 ± 0.09
f1-score,0.35 ± 0.15,0.72 ± 0.09,0.53 ± 0.08
support,36.14 ± 11.48,88.57 ± 17.13,124.71 ± 14.94
